# Biohub Fold 0 aligned detection sweep

Reuse the 5-epoch Fold 0 checkpoint and align inference with training evaluation: no detection TTA, 5 µm pooling, and no ILP. Evaluate thresholds 0.50, 0.57, and 0.65 on the same eight held-out `6bba` clips.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

COMP = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development')
TRAIN = COMP / 'train'
SUPPORT_CANDIDATES = [Path('/kaggle/input/biohub-tracking-support-pack-50ep-v1'), Path('/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1')]
SUPPORT = next((path for path in SUPPORT_CANDIDATES if path.exists()), SUPPORT_CANDIDATES[0])
WORK = Path('/kaggle/working')
REPO = WORK / 'tracking_repo'
INPUT_ROOT = Path('/kaggle/input')
assert TRAIN.exists(), TRAIN
assert SUPPORT.exists(), SUPPORT
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(SUPPORT / 'repo', REPO)

# The support pack contains wheels, so the notebook remains Internet-off.
packages = ['tracksdata', 'zarr', 'pyscipopt', 'geff', 'geff-spec', 'ilpy', 'polars-runtime-32', 'polars', 'blosc2', 'dask', 'imagecodecs', 'pyarrow', 'rustworkx', 'sqlalchemy', 'donfig', 'numcodecs']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', '--find-links', str(SUPPORT / 'wheels'), '--force-reinstall', *packages], check=True)
import polars as pl
assert hasattr(pl, 'Float16'), f'Unsupported Polars: {pl.__version__}'
print('Prepared', REPO)


In [ ]:
import json

def embryo_id(name: str) -> str:
    return name.removesuffix('.zarr').split('_', 1)[0]

samples = sorted(path.name.removesuffix('.zarr') for path in TRAIN.glob('*.zarr'))
embryos = sorted({embryo_id(name) for name in samples})
assert embryos == ['44b6', '6bba'], embryos
held_out = '6bba'
held_out_samples = [name for name in samples if embryo_id(name) == held_out]
pilot_size = 8
pilot_indices = [round(i * (len(held_out_samples) - 1) / (pilot_size - 1)) for i in range(pilot_size)]
pilot_test = [held_out_samples[i] for i in pilot_indices]
assert len(pilot_test) == pilot_size and len(set(pilot_test)) == pilot_size
split = [{
    'held_out_embryo': held_out,
    'train': [name for name in samples if embryo_id(name) != held_out],
    'test': pilot_test,
}]
splits_path = REPO / 'dataset_splits.json'
splits_path.write_text(json.dumps(split, indent=2))
print({'train_clips': len(split[0]['train']), 'pilot_test_clips': len(split[0]['test']), 'held_out': held_out})
print('pilot_test:', pilot_test)


In [ ]:
# Reuse the completed pilot checkpoint; do not retrain for a threshold-only comparison.
all_checkpoints = sorted(INPUT_ROOT.rglob('edge_predictor_best.pth'))
checkpoint_candidates = [p for p in all_checkpoints if 'fold0' in str(p).lower()]
print('Input roots:', sorted(p.name for p in INPUT_ROOT.iterdir()))
print('Checkpoint candidates:', checkpoint_candidates)
assert len(checkpoint_candidates) == 1, checkpoint_candidates
weights_path = REPO / 'weights/unet_transformer/split_0/edge_predictor_best.pth'
weights_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(checkpoint_candidates[0], weights_path)
print('Reused checkpoint:', checkpoint_candidates[0], '->', weights_path, weights_path.stat().st_size)
predict_script = REPO / 'scripts/predict_unet_transformer.py'
predict_text = predict_script.read_text()
predict_text = predict_text.replace('det_tta: bool = True', 'det_tta: bool = False')
predict_text = predict_text.replace('pool_kernel_um: float = 3.0', 'pool_kernel_um: float = 5.0')
predict_script.write_text(predict_text)
assert 'det_tta: bool = False' in predict_text
assert 'pool_kernel_um: float = 5.0' in predict_text
print('Aligned inference: det_tta=False, pool_kernel_um=5.0, use_ilp=False')


In [ ]:
# The training evaluator accepts raw logits > 0.3, equivalent to sigmoid probability 0.5744.
thresholds = [0.50, 0.57, 0.65]
for threshold in thresholds:
    method = f'unet_transformer_thr{round(threshold * 100):02d}'
    print(f'\n=== {method} ===', flush=True)
    predict_cmd = [
        sys.executable, 'scripts/predict_unet_transformer.py',
        '--method', method, '--data-dir', str(TRAIN),
        '--splits', 'dataset_splits.json', '--split', '0',
        '--weights', str(weights_path), '--unet-batch-size', '4',
        '--det-threshold', str(threshold), '--evaluate',
    ]
    subprocess.run(predict_cmd, cwd=REPO, env={**os.environ, 'PYTHONPATH': 'src', 'BIOHUB_DATA_DIR': str(TRAIN)}, check=True)


In [ ]:
# Persist the exact sweep definition for selecting the full-fold inference threshold.
summary = {
    'fold': 0, 'checkpoint_source': 'versavice/biohub-fold0-pilot-5ep',
    'train_embryo': '44b6', 'held_out_embryo': '6bba',
    'thresholds': thresholds,
    'splits': split,
}
(WORK / 'fold0_threshold_sweep_manifest.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
